# Aperture Sampling Convergence Study

This notebook investigates how RMSE converges to theoretical Airy pattern as we increase the number of photons.

For each N_photons, we run multiple trials and compute:
- Mean RMSE
- Standard deviation of RMSE

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.special import j1
from monte_carlo.core import ApertureSimulator
from monte_carlo import metrics

# Set style
sns.set_theme(style="whitegrid", font_scale=1.5)

# Create output directory
import os
output_dir = '../data/convergence'
os.makedirs(output_dir, exist_ok=True)

## Setup Parameters

In [ ]:
# Fixed simulation parameters
wavelength = 0.532  # microns
focal_length = 10.0  # mm
numerical_aperture = 0.1
n_medium = 1.0

# Convergence study parameters
n_photons_list = [1000, 5000, 10000, 50000, 100000, 500000, 1000000]
n_trials = 10  # Number of trials per N_photons

# Airy parameters
airy_radius = 1.22 * wavelength / numerical_aperture
k_theory = 2 * np.pi * numerical_aperture / wavelength

print(f"Wavelength: {wavelength} μm")
print(f"NA: {numerical_aperture}")
print(f"Focal length: {focal_length} mm")
print(f"First Airy zero: {airy_radius:.4f} μm")
print(f"\nN_photons range: {min(n_photons_list)} to {max(n_photons_list)}")
print(f"Trials per N_photons: {n_trials}")

## Compute Theoretical Airy Pattern

In [ ]:
# Create theoretical Airy radial profile
plot_range = 2 * airy_radius
r_bins = np.linspace(0, plot_range, 300)
r_centers = (r_bins[:-1] + r_bins[1:]) / 2
bin_areas = np.pi * (r_bins[1:]**2 - r_bins[:-1]**2)

# Theoretical Airy pattern
kr_theory = k_theory * r_centers
airy_theory = np.ones_like(kr_theory)
nonzero = kr_theory != 0
airy_theory[nonzero] = (2 * j1(kr_theory[nonzero]) / kr_theory[nonzero])**2

print(f"Radial bins: {len(r_bins)}")
print(f"Radial range: [0, {plot_range:.4f}] μm")

## Run Convergence Study

In [ ]:
# Storage for results
rmse_results = {}

for n_photons in n_photons_list:
    print(f"\nRunning N_photons = {n_photons}...")
    rmse_trials = []
    
    for trial in range(n_trials):
        # Create simulator with different seed for each trial
        sim = ApertureSimulator(
            n_photons=n_photons,
            wavelength=wavelength,
            focal_length=focal_length,
            numerical_aperture=numerical_aperture,
            n_medium=n_medium,
            random_seed=trial
        )
        
        # Run simulation
        results = sim.propagate()
        x, y, _ = results['focal_positions']
        
        # Compute radial profile
        r = np.sqrt(x**2 + y**2)
        hist_r, _ = np.histogram(r, bins=r_bins)
        intensity_r = hist_r / bin_areas
        intensity_r = intensity_r / np.max(intensity_r)  # Normalize to peak
        
        # Compute RMSE
        rmse_val = metrics.rmse(intensity_r, airy_theory)
        rmse_trials.append(rmse_val)
    
    # Store results
    rmse_results[n_photons] = {
        'mean': np.mean(rmse_trials),
        'std': np.std(rmse_trials),
        'trials': rmse_trials
    }
    
    print(f"  RMSE: {rmse_results[n_photons]['mean']:.6f} ± {rmse_results[n_photons]['std']:.6f}")

print("\nConvergence study complete!")

## Plot Results

In [ ]:
# Extract data for plotting
n_photons_array = np.array(n_photons_list)
rmse_mean = np.array([rmse_results[n]['mean'] for n in n_photons_list])
rmse_std = np.array([rmse_results[n]['std'] for n in n_photons_list])

# Create figure
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: RMSE vs N_photons (log scale)
ax = axes[0]
ax.errorbar(n_photons_array, rmse_mean, yerr=rmse_std, 
            marker='o', markersize=10, linewidth=3, capsize=5, capthick=2,
            label=f'Aperture Sampling ({n_trials} trials)', color='tab:blue')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Number of Photons')
ax.set_ylabel('RMSE')
ax.set_title('RMSE Convergence vs N_photons', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Add theoretical 1/sqrt(N) scaling line
n_ref = n_photons_array[0]
rmse_ref = rmse_mean[0]
theoretical_scaling = rmse_ref * np.sqrt(n_ref / n_photons_array)
ax.plot(n_photons_array, theoretical_scaling, '--', linewidth=2, 
        alpha=0.5, color='red', label='1/√N scaling')
ax.legend()

# Plot 2: Standard deviation vs N_photons
ax = axes[1]
ax.plot(n_photons_array, rmse_std, marker='s', markersize=10, 
        linewidth=3, color='tab:orange', label='RMSE Standard Deviation')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Number of Photons')
ax.set_ylabel('Standard Deviation of RMSE')
ax.set_title('RMSE Variability vs N_photons', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{output_dir}/aperture_convergence.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nPlot saved to: {output_dir}/aperture_convergence.png")

## Summary Statistics

In [ ]:
print("\n" + "="*80)
print("APERTURE SAMPLING CONVERGENCE STUDY")
print("="*80)
print(f"{'N_photons':<15} {'RMSE (mean)':<20} {'RMSE (std)':<20} {'Relative Std %':<15}")
print("-"*80)
for n in n_photons_list:
    mean_rmse = rmse_results[n]['mean']
    std_rmse = rmse_results[n]['std']
    rel_std = (std_rmse / mean_rmse) * 100 if mean_rmse > 0 else 0
    print(f"{n:<15} {mean_rmse:<20.6f} {std_rmse:<20.6f} {rel_std:<15.2f}")
print("="*80)